## 1. Initialize Project Environment
Import libraries for MSA analysis and conservation scoring.

> **Note:** This notebook uses the MSA result from **Task 1** (`task1_msa_result.clustal`) to analyze conserved regions and compare with the phylogenetic tree.

In [1]:
from __future__ import annotations

import json
import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List

import pandas as pd
from Bio import AlignIO

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

## 2. Define Configuration Parameters
Paths to MSA from Task 1 and conservation analysis options.

In [2]:
@dataclass
class ConservationConfig:
    msa_path: Path = Path("artifacts/task1_msa_result.clustal")  # MSA from Task 1
    export_dir: Path = Path("artifacts")
    conservation_threshold: float = 0.9  # 90% identity = conserved
    min_block_length: int = 10  # Minimum bp for a conserved block

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["msa_path"] = str(info["msa_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = ConservationConfig()
CONFIG.describe()

{'msa_path': 'artifacts/task1_msa_result.clustal',
 'export_dir': 'artifacts',
 'conservation_threshold': 0.9,
 'min_block_length': 10}

In [3]:
# Load MSA from Task 1 (no need to re-run Clustal Omega!)
if not CONFIG.msa_path.exists():
    raise FileNotFoundError(f"MSA not found: {CONFIG.msa_path}. Run Task1 first.")

alignment = AlignIO.read(CONFIG.msa_path, "clustal")
logging.info(
    "Loaded MSA: %d sequences × %d positions",
    len(alignment),
    alignment.get_alignment_length(),
)

print(
    f"Alignment dimensions: {len(alignment)} seqs × {alignment.get_alignment_length()} bp"
)
print(f"\nSequences in MSA:")
for rec in alignment:
    print(f"  {rec.id}")
print(f"\nFirst 80 columns of alignment:\n{alignment[:, :80]}")

[INFO] Loaded MSA: 10 sequences × 1859 positions


Alignment dimensions: 10 seqs × 1859 bp

Sequences in MSA:
  NM_000546.6
  NM_011640.3
  NM_131327.2
  NM_001003210.1
  NM_213824.3
  NM_174201.2
  NM_205264.1
  NM_030989.3
  XM_016931470.3
  NM_001047151.2

First 80 columns of alignment:
Alignment with 10 rows and 80 columns
------CTCAAAAGTCT---------------------------...TGC NM_000546.6
TTTCCCCTCCCACGTGCTCACCCTGGCTAAAGTTCTGTAGCTTC...GAC NM_011640.3
--------------------------------------------...--- NM_131327.2
--------------------------------------------...--- NM_001003210.1
---------AAAAGTCC---------------------------...TGC NM_213824.3
---------TAAAGTCC---------------------------...CGC NM_174201.2
--------------------------------------------...--- NM_205264.1
---------------------------------TCTGAAGCTCC...GAC NM_030989.3
------CTCAAAAGTCT---------------------------...TGC XM_016931470.3
--------------------------------------------...--- NM_001047151.2


## 3. Compute Conservation Scores
Calculate per-position conservation and identify conserved regions.

In [4]:
def compute_conservation(
    alignment, threshold: float = 0.9, exclude_gaps: bool = True
) -> pd.DataFrame:
    """Calculate conservation score for each position.

    Args:
        alignment: MSA alignment object
        threshold: Minimum fraction to consider position conserved
        exclude_gaps: If True, don't count gap-only columns as conserved
    """
    positions = []
    aln_len = alignment.get_alignment_length()

    for pos in range(aln_len):
        column = alignment[:, pos]
        counts = pd.Series(list(column)).value_counts()
        top_base = counts.index[0]
        score = counts.iloc[0] / counts.sum()

        # Don't count positions where the "conserved" base is a gap
        is_conserved = score >= threshold
        if exclude_gaps and top_base == "-":
            is_conserved = False

        positions.append(
            {
                "position": pos,
                "top_base": top_base,
                "score": score,
                "conserved": is_conserved,
            }
        )

    return pd.DataFrame(positions)


conservation_df = compute_conservation(alignment, CONFIG.conservation_threshold)
print(f"Total alignment positions: {len(conservation_df)}")
print(
    f"Conserved positions (≥{CONFIG.conservation_threshold}, excluding gaps): {conservation_df['conserved'].sum()}"
)
print(f"Mean conservation score: {conservation_df['score'].mean():.3f}")
conservation_df.head(20)

Total alignment positions: 1859
Conserved positions (≥0.9, excluding gaps): 601
Mean conservation score: 0.756


,position,top_base,score,conserved
0,0,-,0.9,False
1,1,-,0.9,False
2,2,-,0.9,False
3,3,-,0.9,False
4,4,-,0.9,False
5,5,-,0.9,False
6,6,-,0.7,False
7,7,-,0.7,False
8,8,-,0.7,False
9,9,-,0.5,False


## 4. Find Conserved Blocks
Identify contiguous stretches of conserved positions (functionally important regions).

In [5]:
def find_conserved_blocks(
    conservation_df: pd.DataFrame, min_length: int = 10
) -> List[Dict]:
    """Find consecutive runs of conserved positions."""
    blocks = []
    in_block = False
    block_start = 0

    for i, row in conservation_df.iterrows():
        if row["conserved"] and not in_block:
            in_block = True
            block_start = row["position"]
        elif not row["conserved"] and in_block:
            in_block = False
            block_len = row["position"] - block_start
            if block_len >= min_length:
                blocks.append(
                    {
                        "start": block_start,
                        "end": row["position"] - 1,
                        "length": block_len,
                    }
                )

    # Handle last block
    if in_block:
        block_len = len(conservation_df) - block_start
        if block_len >= min_length:
            blocks.append(
                {
                    "start": block_start,
                    "end": len(conservation_df) - 1,
                    "length": block_len,
                }
            )

    return blocks


conserved_blocks = find_conserved_blocks(conservation_df, CONFIG.min_block_length)
print(
    f"Found {len(conserved_blocks)} conserved blocks (≥{CONFIG.min_block_length} bp):"
)
for block in conserved_blocks[:10]:
    print(f"  Position {block['start']}-{block['end']} ({block['length']} bp)")

Found 5 conserved blocks (≥10 bp):
  Position 788-801 (14 bp)
  Position 842-855 (14 bp)
  Position 1037-1047 (11 bp)
  Position 1122-1131 (10 bp)
  Position 1151-1161 (11 bp)


## 5. Summarize Conservation and Compare with Tree
Generate summary statistics and relate conservation to phylogenetic distances.

In [6]:
def summarize_conservation(
    conservation_df: pd.DataFrame, conserved_blocks: List[Dict], threshold: float
) -> Dict:
    """Generate summary statistics."""
    total_pos = len(conservation_df)
    conserved_count = conservation_df["conserved"].sum()
    conserved_pct = 100 * conserved_count / total_pos
    mean_score = conservation_df["score"].mean()

    # Exclude gap-dominated positions for biological relevance
    non_gap = conservation_df[conservation_df["top_base"] != "-"]
    non_gap_conserved = non_gap["conserved"].sum()

    summary = {
        "total_positions": total_pos,
        "conserved_positions": int(conserved_count),
        "conserved_percent": round(conserved_pct, 1),
        "mean_conservation_score": round(mean_score, 3),
        "non_gap_positions": len(non_gap),
        "non_gap_conserved": int(non_gap_conserved),
        "num_conserved_blocks": len(conserved_blocks),
        "threshold_used": threshold,
    }

    # Top 3 longest conserved blocks
    top_blocks = sorted(conserved_blocks, key=lambda x: x["length"], reverse=True)[:3]
    summary["top_blocks"] = [
        f"{b['start']}-{b['end']} ({b['length']}bp)" for b in top_blocks
    ]

    return summary


summary_stats = summarize_conservation(
    conservation_df, conserved_blocks, CONFIG.conservation_threshold
)

print("=== Conservation Summary ===\n")
for k, v in summary_stats.items():
    print(f"  {k}: {v}")

=== Conservation Summary ===

  total_positions: 1859
  conserved_positions: 601
  conserved_percent: 32.3
  mean_conservation_score: 0.756
  non_gap_positions: 1473
  non_gap_conserved: 601
  num_conserved_blocks: 5
  threshold_used: 0.9
  top_blocks: ['788-801 (14bp)', '842-855 (14bp)', '1037-1047 (11bp)']


## 6. Export Results
Save conservation data to artifacts.

In [7]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save conservation scores CSV
cons_path = EXPORT_DIR / "task3_conservation_scores.csv"
conservation_df.to_csv(cons_path, index=False)
print(f"[OK] Conservation scores saved to: {cons_path.resolve()}")

# Save summary as JSON
summary_path = EXPORT_DIR / "task3_conservation_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_stats, f, indent=2)
print(f"[OK] Conservation summary saved to: {summary_path.resolve()}")

print(f"\nArtifacts saved to {EXPORT_DIR.resolve()}")

[OK] Conservation scores saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task3_conservation_scores.csv
[OK] Conservation summary saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task3_conservation_summary.json

Artifacts saved to /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts
